# Line ratios 

In [22]:
import numpy as np
import pandas as pd

from astroExplain.spectra import astronomy
from sdss.metadata import MetaData

meta = MetaData()
pd.set_option("display.max_columns", None)

# Custom functions

# Config

## Constants

In [23]:
se_cols = ["mse", "mse_filter_250", "mse_97", "mse_filter_250_97"]
se_rank_cols = [f"rank_{col}" for col in se_cols]
# ----------------------------------------------
rse_cols = [f"{col}_rel" for col in se_cols]
rse_rank_cols = [f"rank_{col}_rel" for col in se_cols]
# ----------------------------------------------
se_family = ["mse", "mse_97", "mse_filter_250", "mse_filter_250_97"]
rse_family = [f"{col}_rel" for col in se_family]

## Directories

In [24]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

# Data

In [25]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave * 0.1

spectra = np.load(f"{spectra_dir}/spectra_imputed.npy", mmap_mode="r")

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

meta_with_lines_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_lines.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(f"{spectra_dir}/ids_imputing.npy", mmap_mode="r")

# lines_df = pd.read_csv(
#     f"{spectra_dir}/meta_data/lines_EdgarOrtiz.csv",
#     index_col="specObjID",
# )

In [26]:
meta_with_lines_df.head(3)
# final_meta_df.head(3)

,mjd,plate,fiberid,run2d,ra,dec,z,zErr,zWarning,class,subClass,z_noqso,zErr_noqso,zWarning_noqso,targetType,programname,instrument,snMedian,ABSSB,BROAD,ebv,indexArray,oii_3726_flux,oii_3726_flux_err,oii_3729_flux,oii_3729_flux_err,neiii_3869_flux,neiii_3869_flux_err,h_delta_flux,h_delta_flux_err,h_gamma_flux,h_gamma_flux_err,oiii_4363_flux,oiii_4363_flux_err,h_beta_flux,h_beta_flux_err,oiii_4959_flux,oiii_4959_flux_err,oiii_5007_flux,oiii_5007_flux_err,hei_5876_flux,hei_5876_flux_err,oi_6300_flux,oi_6300_flux_err,nii_6548_flux,nii_6548_flux_err,h_alpha_flux,h_alpha_flux_err,nii_6584_flux,nii_6584_flux_err,sii_6717_flux,sii_6717_flux_err,sii_6731_flux,sii_6731_flux_err,ariii7135_flux,ariii7135_flux_err,oii_flux,oii_flux_err,oiii_flux,oiii_flux_err
specobjid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1874756710606858240,52976,1665,485,26,49.861444,41.540485,0.017909,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,94.13022,BROADLINE,BROADLINE,0.138756,2.0,0.0,-7.832294e-01,30418890.00,1.842604e+06,-40.570890,10.948140,21.51809,13.21366,49.39598,15.19424,-77.73644,16.07351,109.63340,17.43791,-49.362380,19.09110,176.81650,17.88523,-201.0114,16.10323,34.87749,22.67058,49.30014,11.300770,109.7575,35.74986,148.70060,34.08569,3.150983,31.61602,-2.93755,31.82735,-103.19390,23.14498,0.0000,1.107654,174.70850,21.45677
1874765781577787392,52976,1665,518,26,49.918970,41.548643,0.021434,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,92.63803,BROADLINE,BROADLINE,0.141073,3.0,-603451.0,1.139662e+06,30924.64,3.962919e+04,-1.672035,7.612301,30.98863,11.73072,26.67140,12.99125,-20.94539,10.47208,41.71867,14.98429,2.058654,12.20163,72.43165,12.41198,-289.6941,13.68605,-13.75883,14.15635,29.28021,6.114465,132.8577,23.17760,88.31586,18.44262,-44.478840,16.03369,18.97761,16.29948,-36.61877,14.79838,722.0763,1365.593000,67.69716,12.80094
1874776226938251264,52976,1665,556,26,50.251707,41.562451,0.015699,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.91905,BROADLINE,BROADLINE,0.142785,5.0,0.0,-7.819291e-01,0.00,-7.819291e-01,-42.426140,9.273634,16.98602,11.56076,51.72350,12.93845,-121.21380,13.62333,169.26810,14.49883,-80.583990,15.25546,147.23940,15.11998,-346.5715,12.84481,21.52090,17.46943,45.43762,7.907939,261.2984,25.16728,137.05040,23.85214,27.470890,23.79437,-16.98146,23.95586,-95.06487,18.50055,0.0000,1.105815,150.70450,14.26691


## Line to metadata

```python
meta_with_lines_df = final_meta_df.join(lines_df, how="left")
meta_with_lines_df.to_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_lines.csv.gz",
    index=True,
    compression="gzip"
)
```

# EDA fluxes

In [29]:
lines_cols = [
    "oii_3726_flux",
    "oii_3729_flux",
    "h_beta_flux",
    "oiii_4959_flux",
    "oiii_5007_flux",
    "oiii_flux",
    "h_alpha_flux",
    "nii_6548_flux",
    "nii_6584_flux",
    "sii_6717_flux",
    "sii_6731_flux",
]
meta_with_lines_df[lines_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
oii_3726_flux,725949.0,3.410111e+06,1.034478e+09,-2.128498e+11,3.048903,12.341530,29.932920,3.091597e+11
oii_3729_flux,725949.0,1.190026e+11,1.015050e+14,-1.025833e+14,3.492142,13.768670,32.727520,8.648484e+16
h_beta_flux,725949.0,1.240318e+06,8.289394e+07,-1.084135e+10,4.559324,13.842160,37.398280,2.328642e+10
oiii_4959_flux,725949.0,6.457281e+05,7.961922e+07,-7.504712e+08,0.698237,3.610237,8.079025,4.203237e+10
oiii_5007_flux,725949.0,1.938375e+06,2.414195e+08,-2.570217e+08,4.868986,10.371990,22.046410,1.298335e+11
oiii_flux,725949.0,1.937335e+06,2.409359e+08,-3.532498e+08,5.311331,10.956190,22.829190,1.293918e+11
h_alpha_flux,725949.0,4.999619e+06,5.087292e+08,-1.220816e+11,13.882550,52.394540,156.187700,3.234893e+11
nii_6548_flux,725949.0,4.368884e+05,6.347367e+07,-1.504113e+08,3.317939,9.007028,20.820210,5.080869e+10
nii_6584_flux,725949.0,1.317756e+06,1.914513e+08,-4.536753e+08,10.007670,27.167270,62.798560,1.532508e+11
sii_6717_flux,725949.0,5.110252e+05,4.311879e+08,-2.631785e+11,3.637836,13.267800,31.587770,1.520267e+11


In [30]:
# Compute deciles (10th to 90th percentiles) for each column
deciles = np.arange(0, 1.01, 0.1)
decile_df = meta_with_lines_df[lines_cols].quantile(q=deciles).T
decile_df.columns = [f"{int(q * 100)}th" for q in deciles]
decile_df.T

,oii_3726_flux,oii_3729_flux,h_beta_flux,oiii_4959_flux,oiii_5007_flux,oiii_flux,h_alpha_flux,nii_6548_flux,nii_6584_flux,sii_6717_flux,sii_6731_flux
0th,-2.128498e+11,-1.025833e+14,-1.084135e+10,-7.504712e+08,-2.570217e+08,-3.532498e+08,-1.220816e+11,-1.504113e+08,-4.536753e+08,-2.631785e+11,-9.089749e+11
10th,-1.199457e+00,-1.313241e+00,7.741553e-01,-1.835093e+00,1.516692e+00,1.898060e+00,4.018114e+00,6.046451e-01,1.823749e+00,-7.769778e-01,-1.684574e+00
20th,1.445959e+00,1.695242e+00,3.310159e+00,3.496830e-02,3.824329e+00,4.294110e+00,1.001189e+01,2.285004e+00,6.892098e+00,2.066326e+00,8.778535e-01
30th,4.722694e+00,5.358730e+00,5.926755e+00,1.304135e+00,5.895282e+00,6.319991e+00,1.843547e+01,4.351354e+00,1.312469e+01,5.356861e+00,3.356872e+00
40th,8.307881e+00,9.377713e+00,9.239538e+00,2.442752e+00,7.985792e+00,8.464136e+00,3.131520e+01,6.489165e+00,1.957282e+01,9.024865e+00,6.128862e+00
50th,1.234153e+01,1.376867e+01,1.384216e+01,3.610237e+00,1.037199e+01,1.095619e+01,5.239454e+01,9.007028e+00,2.716727e+01,1.326780e+01,9.304872e+00
60th,1.740768e+01,1.923905e+01,2.045092e+01,4.957425e+00,1.344447e+01,1.413062e+01,8.292303e+01,1.236977e+01,3.731010e+01,1.868218e+01,1.328477e+01
70th,2.467899e+01,2.704617e+01,3.039259e+01,6.775065e+00,1.822420e+01,1.897452e+01,1.261319e+02,1.731655e+01,5.223070e+01,2.629834e+01,1.889428e+01
80th,3.728767e+01,4.063131e+01,4.678089e+01,9.956496e+00,2.794216e+01,2.874512e+01,1.965176e+02,2.552122e+01,7.697788e+01,3.863270e+01,2.795252e+01
90th,6.850883e+01,7.472080e+01,8.284060e+01,2.015665e+01,6.014325e+01,6.157294e+01,3.466838e+02,4.281620e+01,1.291436e+02,6.464570e+01,4.725372e+01


In [31]:
n_nans = meta_with_lines_df[lines_cols].isna().sum()
n_negatives = (meta_with_lines_df[lines_cols] < 0).sum()
n_zeros = (meta_with_lines_df[lines_cols] == 0).sum()
temp_df = pd.DataFrame(
    {"n_nans": n_nans, "n_negatives": n_negatives, "n_zeros": n_zeros}
)
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_nans,n_negatives,n_zeros
0,oii_3726_flux,1452,96253,15957
1,oii_3729_flux,1452,95641,14516
2,h_beta_flux,1452,53512,863
3,oiii_4959_flux,1452,142411,833
4,oiii_5007_flux,1452,36342,870
5,oiii_flux,1452,38707,833
6,h_alpha_flux,1452,20631,2114
7,nii_6548_flux,1452,37780,1680
8,nii_6584_flux,1452,37780,1680
9,sii_6717_flux,1452,91004,2177


# Data prep to compute ratios

In [ ]:
# replace 0 with NaN
meta_with_lines_df[lines_cols] = meta_with_lines_df[lines_cols].replace(0, np.nan)
# replace negative fluxes numeric values with NaN
meta_with_lines_df[lines_cols] = meta_with_lines_df[lines_cols].where(
    meta_with_lines_df[lines_cols] >= 0, np.nan
)
n_zeros = (meta_with_lines_df[lines_cols] == 0).sum()
n_negatives = (meta_with_lines_df[lines_cols] < 0).sum()
temp_df = pd.DataFrame({"n_negatives": n_negatives, "n_zeros": n_zeros})
temp_df.index.name = "line_flux_column"
temp_df = temp_df.reset_index()
temp_df

,line_flux_column,n_negatives,n_zeros
0,oii_3726_flux,0,0
1,oii_3729_flux,0,0
2,h_beta_flux,0,0
3,oiii_4959_flux,0,0
4,oiii_5007_flux,0,0
5,oiii_flux,0,0
6,h_alpha_flux,0,0
7,nii_6548_flux,0,0
8,nii_6584_flux,0,0
9,sii_6717_flux,0,0


# Line ratios

In [33]:
ratios_df, cols_ratios = astronomy.compute_emission_line_ratios(meta_with_lines_df)
ratios_df.head()

,mjd,plate,fiberid,run2d,ra,dec,z,zErr,zWarning,class,subClass,z_noqso,zErr_noqso,zWarning_noqso,targetType,programname,instrument,snMedian,ABSSB,BROAD,ebv,indexArray,oii_3726_flux,oii_3726_flux_err,oii_3729_flux,oii_3729_flux_err,neiii_3869_flux,neiii_3869_flux_err,h_delta_flux,h_delta_flux_err,h_gamma_flux,h_gamma_flux_err,oiii_4363_flux,oiii_4363_flux_err,h_beta_flux,h_beta_flux_err,oiii_4959_flux,oiii_4959_flux_err,oiii_5007_flux,oiii_5007_flux_err,hei_5876_flux,hei_5876_flux_err,oi_6300_flux,oi_6300_flux_err,nii_6548_flux,nii_6548_flux_err,h_alpha_flux,h_alpha_flux_err,nii_6584_flux,nii_6584_flux_err,sii_6717_flux,sii_6717_flux_err,sii_6731_flux,sii_6731_flux_err,ariii7135_flux,ariii7135_flux_err,oii_flux,oii_flux_err,oiii_flux,oiii_flux_err,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
specobjid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1874756710606858240,52976,1665,485,26,49.861444,41.540485,0.017909,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,94.13022,BROADLINE,BROADLINE,0.138756,2.0,NaN,-7.832294e-01,30418890.00,1.842604e+06,-40.570890,10.948140,21.518090,13.21366,49.39598,15.19424,-77.73644,16.07351,109.63340,17.43791,NaN,19.09110,176.81650,17.88523,-201.0114,16.10323,34.87749,22.67058,49.30014,11.300770,109.7575,35.74986,148.70060,34.08569,3.150983,31.61602,NaN,31.82735,-103.19390,23.14498,0.0000,1.107654,174.70850,21.45677,1.001132,1.354810,1.612798,NaN,0.075701,NaN,NaN
1874765781577787392,52976,1665,518,26,49.918970,41.548643,0.021434,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,92.63803,BROADLINE,BROADLINE,0.141073,3.0,NaN,1.139662e+06,30924.64,3.962919e+04,-1.672035,7.612301,30.988630,11.73072,26.67140,12.99125,-20.94539,10.47208,41.71867,14.98429,2.058654,12.20163,72.43165,12.41198,-289.6941,13.68605,-13.75883,14.15635,29.28021,6.114465,132.8577,23.17760,88.31586,18.44262,NaN,16.03369,18.97761,16.29948,-36.61877,14.79838,722.0763,1365.593000,67.69716,12.80094,3.184610,0.664740,1.736193,NaN,0.416946,NaN,NaN
1874776226938251264,52976,1665,556,26,50.251707,41.562451,0.015699,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.91905,BROADLINE,BROADLINE,0.142785,5.0,NaN,-7.819291e-01,NaN,-7.819291e-01,-42.426140,9.273634,16.986020,11.56076,51.72350,12.93845,-121.21380,13.62333,169.26810,14.49883,NaN,15.25546,147.23940,15.11998,-346.5715,12.84481,21.52090,17.46943,45.43762,7.907939,261.2984,25.16728,137.05040,23.85214,27.470890,23.79437,NaN,23.95586,-95.06487,18.50055,0.0000,1.105815,150.70450,14.26691,1.543695,0.524498,0.869859,NaN,0.219705,NaN,NaN
1874654181147568128,52976,1665,112,26,50.336818,41.460050,0.015284,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.70859,BROADLINE,BROADLINE,0.146074,6.0,NaN,-7.849703e-01,NaN,-7.849703e-01,-23.003590,11.276780,-8.040654,13.05702,38.61938,14.63279,-129.42430,15.76263,108.02560,14.94735,NaN,15.86833,114.72650,16.02149,-349.5778,13.19907,21.59566,18.24788,35.81715,8.626451,185.5176,26.94842,108.03280,26.01934,7.737221,25.64335,NaN,25.80392,-88.79891,19.41240,0.0000,1.110116,111.87690,17.13640,1.717348,0.582332,1.062031,NaN,0.260967,NaN,NaN
1874775677182437376,52976,1665,554,26,50.211261,41.600418,0.017639,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.41394,BROADLINE,BROADLINE,0.140459,7.0,NaN,-7.952285e-01,NaN,-7.952285e-01,-17.693380,10.373580,-11.948000,11.96610,-10.20461,13.24161,-75.50719,13.90316,107.99700,14.96747,NaN,15.86618,156.05990,14.82815,-478.1284,13.02057,10.05682,18.21331,57.57146,8.112593,170.9763,25.97289,173.64880,24.46943,1.600154,25.62592,NaN,25.77136,-97.76904,18.78243,0.0000,1.124623,154.58710,14.79067,1.583158,1.015631,1.445039,NaN,0.153144,NaN,NaN


In [34]:
# Compute percentiles (45th to 55th percentiles) for each column
percentiles = np.arange(0.45, 0.55, 0.05)
# Create a DataFrame of percentiles for each column in fluxes_df[lines_cols]
ratios_percentiles_df = ratios_df[cols_ratios].quantile(q=percentiles).T
# Optionally rename the rows as "10th", "20th", etc.
ratios_percentiles_df.columns = [f"{int(q * 100)}th" for q in percentiles]
# Display the result
ratios_percentiles_df

,45th,50th,55th
balmer_decrement,3.768733,3.914880,4.063091
nii_to_halpha,0.420978,0.456375,0.499100
oiii_to_hbeta,0.643393,0.743624,0.861812
oiii_to_oii,0.677151,0.735276,0.802389
o3n2_index,0.087218,0.134590,0.184831
sii_to_halpha,0.342261,0.359278,0.378422
sii_density_ratio,1.369007,1.400479,1.432541


# Save line ratios

In [35]:
ratios_df.shape

(727401, 67)

In [ ]:
meta_cols = final_meta_df.columns.to_list()
line_cols_err = [f"{col}_err" for col in lines_cols]

In [42]:
final_ratios = ratios_df[meta_cols + lines_cols + line_cols_err + cols_ratios].copy()
final_ratios.head()

,mjd,plate,fiberid,run2d,ra,dec,z,zErr,zWarning,class,subClass,z_noqso,zErr_noqso,zWarning_noqso,targetType,programname,instrument,snMedian,ABSSB,BROAD,ebv,indexArray,oii_3726_flux,oii_3729_flux,h_beta_flux,oiii_4959_flux,oiii_5007_flux,oiii_flux,h_alpha_flux,nii_6548_flux,nii_6584_flux,sii_6717_flux,sii_6731_flux,oii_3726_flux_err,oii_3729_flux_err,h_beta_flux_err,oiii_4959_flux_err,oiii_5007_flux_err,oiii_flux_err,h_alpha_flux_err,nii_6548_flux_err,nii_6584_flux_err,sii_6717_flux_err,sii_6731_flux_err,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
specobjid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1874756710606858240,52976,1665,485,26,49.861444,41.540485,0.017909,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,94.13022,BROADLINE,BROADLINE,0.138756,2.0,NaN,30418890.00,109.63340,NaN,176.81650,174.70850,109.7575,49.30014,148.70060,3.150983,NaN,-7.832294e-01,1.842604e+06,17.43791,19.09110,17.88523,21.45677,35.74986,11.300770,34.08569,31.61602,31.82735,1.001132,1.354810,1.612798,NaN,0.075701,NaN,NaN
1874765781577787392,52976,1665,518,26,49.918970,41.548643,0.021434,0.000004,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,92.63803,BROADLINE,BROADLINE,0.141073,3.0,NaN,30924.64,41.71867,2.058654,72.43165,67.69716,132.8577,29.28021,88.31586,NaN,18.97761,1.139662e+06,3.962919e+04,14.98429,12.20163,12.41198,12.80094,23.17760,6.114465,18.44262,16.03369,16.29948,3.184610,0.664740,1.736193,NaN,0.416946,NaN,NaN
1874776226938251264,52976,1665,556,26,50.251707,41.562451,0.015699,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.91905,BROADLINE,BROADLINE,0.142785,5.0,NaN,NaN,169.26810,NaN,147.23940,150.70450,261.2984,45.43762,137.05040,27.470890,NaN,-7.819291e-01,-7.819291e-01,14.49883,15.25546,15.11998,14.26691,25.16728,7.907939,23.85214,23.79437,23.95586,1.543695,0.524498,0.869859,NaN,0.219705,NaN,NaN
1874654181147568128,52976,1665,112,26,50.336818,41.460050,0.015284,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.70859,BROADLINE,BROADLINE,0.146074,6.0,NaN,NaN,108.02560,NaN,114.72650,111.87690,185.5176,35.81715,108.03280,7.737221,NaN,-7.849703e-01,-7.849703e-01,14.94735,15.86833,16.02149,17.13640,26.94842,8.626451,26.01934,25.64335,25.80392,1.717348,0.582332,1.062031,NaN,0.260967,NaN,NaN
1874775677182437376,52976,1665,554,26,50.211261,41.600418,0.017639,0.000006,0,GALAXY,BROADLINE,0,0,0,SCIENCE,perseus,SDSS,90.41394,BROADLINE,BROADLINE,0.140459,7.0,NaN,NaN,107.99700,NaN,156.05990,154.58710,170.9763,57.57146,173.64880,1.600154,NaN,-7.952285e-01,-7.952285e-01,14.96747,15.86618,14.82815,14.79067,25.97289,8.112593,24.46943,25.62592,25.77136,1.583158,1.015631,1.445039,NaN,0.153144,NaN,NaN


In [48]:
final_ratios.to_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop_with_ratios.csv.gz",
    index=True,
    compression="gzip",
)

# Common anomalies

In [43]:
overview_common_dict = {
    # row 1
    "narrow_line": 734111514142730240,
    "broad_line_large_OIII": 1633733043925575680,
    # row 2
    "broad_emission_dips_OIII_half": 1192355533059811328,
    "star_forming_step_blue_slope": 1959124163192973312,
    # row 3
    "blue_bump_emission": 531492683672217600,
    "passive_star": 1780176998165932032,
    # row 4
    "spike": 1413149843194406912,
    "noise_forest": 637325355518027776,
}

In [44]:
cols_ratios = [
    "balmer_decrement",
    "nii_to_halpha",
    "oiii_to_hbeta",
    "oiii_to_oii",
    "o3n2_index",
    "sii_to_halpha",
    "sii_density_ratio",
]
final_ratios.loc[734111514142730240, cols_ratios]

balmer_decrement      3.29336
nii_to_halpha         0.05016
oiii_to_hbeta        3.990424
oiii_to_oii               NaN
o3n2_index           1.900664
sii_to_halpha        0.163502
sii_density_ratio    1.362553
Name: 734111514142730240, dtype: object

In [45]:
ratios_dict = {
    "specobjid": [],
    "name": [],
    "balmer_decrement": [],
    "nii_to_halpha": [],
    "oiii_to_hbeta": [],
    "oiii_to_oii": [],
    "o3n2_index": [],
    "sii_to_halpha": [],
    "sii_density_ratio": [],
}


for title, specid in overview_common_dict.items():
    ratios_dict["name"].append(title)
    ratios_dict["specobjid"].append(specid)
    for col in cols_ratios:
        try:
            ratios_dict[col].append(ratios_df.loc[specid, col])
        except KeyError:
            ratios_dict[col].append(np.nan)

In [46]:
common_anomalies_ratios_df = pd.DataFrame(ratios_dict)
common_anomalies_ratios_df.to_clipboard()
common_anomalies_ratios_df

,specobjid,name,balmer_decrement,nii_to_halpha,oiii_to_hbeta,oiii_to_oii,o3n2_index,sii_to_halpha,sii_density_ratio
0,734111514142730240,narrow_line,3.293360,0.050160,3.990424,NaN,1.900664,0.163502,1.362553
1,1633733043925575680,broad_line_large_OIII,3.891876,0.204359,10.550430,11.752099,1.712876,0.217171,1.109206
2,1192355533059811328,broad_emission_dips_OIII_half,5.013954,0.599002,8.754367,7.460670,1.164797,0.344767,1.184883
3,1959124163192973312,star_forming_step_blue_slope,3.437888,0.182478,1.409797,0.595381,0.887945,0.278217,1.376201
4,531492683672217600,blue_bump_emission,3.584317,0.371369,0.517071,0.429446,0.143744,0.333807,1.389607
5,1780176998165932032,passive_star,NaN,NaN,NaN,0.123804,NaN,NaN,NaN
6,1413149843194406912,spike,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,637325355518027776,noise_forest,5.763929,0.055846,0.861052,0.411711,1.188040,NaN,NaN


In [47]:
class_common_anomalies_dict = {
    "specobjid": [],
    "name": [],
    "class": [],
}

for title, specid in overview_common_dict.items():

    class_common_anomalies_dict["name"].append(title)
    class_common_anomalies_dict["specobjid"].append(specid)
    try:
        class_ = final_meta_df.loc[specid, "subClass"]
        class_common_anomalies_dict["class"].append(class_)
    except KeyError:
        class_common_anomalies_dict["class"].append(np.nan)

class_common_anomalies_df = pd.DataFrame(class_common_anomalies_dict)
class_common_anomalies_df.to_clipboard()
class_common_anomalies_df

,specobjid,name,class
0,734111514142730240,narrow_line,STARBURST
1,1633733043925575680,broad_line_large_OIII,STARBURST
2,1192355533059811328,broad_emission_dips_OIII_half,AGN BROADLINE
3,1959124163192973312,star_forming_step_blue_slope,STARBURST
4,531492683672217600,blue_bump_emission,STARFORMING
5,1780176998165932032,passive_star,NaN
6,1413149843194406912,spike,NaN
7,637325355518027776,noise_forest,NaN
